# Fiap - Pós Tech IA para Devs

## Tech Challenger Fase 03 - OpenAI

### Sessão 01- Configuração do Ambiente e Diagnóstico
- Setup e Verificação de Credenciais

In [1]:
import google.generativeai as genai
import os
from dotenv import load_dotenv

load_dotenv()
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

print("Modelos disponíveis para sua chave:")
try:
    for m in genai.list_models():
        if 'generateContent' in m.supported_generation_methods:
            print(f"ID: {m.name} | Display Name: {m.display_name}")
except Exception as e:
    print(f"Erro ao listar: {e}")

d:\REPOS-GITHUB-PUBLICO\fiap-pos-tech-ia-para-devs-tech-challenge-fase-3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\User\AppData\Local\Temp\ipykernel_15060\1368988868.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Modelos disponíveis para sua chave:
ID: models/gemini-2.5-flash | Display Name: Gemini 2.5 Flash
ID: models/gemini-2.5-pro | Display Name: Gemini 2.5 Pro
ID: models/gemini-2.0-flash | Display Name: Gemini 2.0 Flash
ID: models/gemini-2.0-flash-001 | Display Name: Gemini 2.0 Flash 001
ID: models/gemini-2.0-flash-lite-001 | Display Name: Gemini 2.0 Flash-Lite 001
ID: models/gemini-2.0-flash-lite | Display Name: Gemini 2.0 Flash-Lite
ID: models/gemini-2.5-flash-preview-tts | Display Name: Gemini 2.5 Flash Preview TTS
ID: models/gemini-2.5-pro-preview-tts | Display Name: Gemini 2.5 Pro Preview TTS
ID: models/gemma-3-1b-it | Display Name: Gemma 3 1B
ID: models/gemma-3-4b-it | Display Name: Gemma 3 4B
ID: models/gemma-3-12b-it | Display Name: Gemma 3 12B
ID: models/gemma-3-27b-it | Display Name: Gemma 3 27B
ID: models/gemma-3n-e4b-it | Display Name: Gemma 3n E4B
ID: models/gemma-3n-e2b-it | Display Name: Gemma 3n E2B
ID: models/gemma-4-26b-a4b-it | Display Name: Gemma 4 26B A4B IT
ID: models/

### Sessão 02 - Arquitetura do Roteador (Inteligência de Decisão)
- Construção do Roteador de Intenções (Decision Maker)

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import Literal

# 1. Estrutura de saída (Pydantic)
class RouteQuery(BaseModel):
    # Adicionamos "fora_de_escopo" na lista de opções permitidas
    topic: Literal["clinico", "triagem", "seguranca", "fora_de_escopo"] = Field(
        description="Classifica a dúvida. Use 'fora_de_escopo' para assuntos não médicos."
    )

# 2. Configuração com o modelo correto
# llm = ChatGoogleGenerativeAI(model="gemini-flash-latest", temperature=0)
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview", temperature=0)
# llm = ChatGoogleGenerativeAI(model="models/gemini-1.5-flash-8b", temperature=0)

# 3. Prompt de Roteamento
system_prompt = """Você é um especialista em triagem de saúde da mulher.
Classifique a entrada:
- 'clinico': Dúvidas sobre exames, DIU, ciclo, menopausa.
- 'triagem': Urgências como dor forte, febre alta, sangramento intenso.
- 'seguranca': Relatos de medo, agressão ou violência doméstica.
- 'fora_de_escopo': Qualquer assunto que NÃO seja saúde ou segurança da mulher."""

route_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}"),
])

# 4. Construção da Chain com Output Estruturado
router_chain = route_prompt | llm.with_structured_output(RouteQuery)

print("✅ Roteador atualizado com Gemini Flash Latest!")

✅ Roteador atualizado com Gemini Flash Latest!


### Sessão 03 - Validação e Testes de Fluxo
- Testes Unitários de Classificação

![árvore de decisão](decision-tree.jpg)

In [7]:
# Teste 1: Segurança
print(f"Teste 1: {router_chain.invoke({'question': 'Tenho medo do meu parceiro, ele é agressivo.'})}")

Teste 1: topic='seguranca'


In [8]:
# Teste 2: Clínico
print(f"Teste 2: {router_chain.invoke({'question': 'Como funciona o DIU de cobre?'})}")

Teste 2: topic='clinico'


In [9]:
# Teste 3: Triagem
print(f"Teste 3: {router_chain.invoke({'question': 'Estou com uma dor pélvica insuportável e febre.'})}")

Teste 3: topic='triagem'


### Sessão 04 - Ingestão da Base de Conhecimento (Protocolos)
- Objetivo: Criar os documentos técnicos que servirão de fonte para o RAG.

In [10]:
# ==============================================================================
# SEÇÃO 4: INGESTÃO DA BASE DE CONHECIMENTO (PROTOCOLOS)
# Objetivo: Criar os documentos técnicos que servirão de fonte para o RAG.
# ==============================================================================

from langchain_core.documents import Document

# Lista de protocolos simulados para o hospital especializado
protocolos_hospitalares = [
    Document(
        page_content="""PROTOCOLO DE PRÉ-NATAL (BAIXO RISCO):
        As consultas devem ser mensais até a 28ª semana. 
        Exames obrigatórios: Hemograma, Tipagem ABO/Rh, Glicemia, HIV e Sífilis.
        Sinais de alerta: Sangramento vaginal ou ausência de movimentos fetais.""",
        metadata={"categoria": "obstetricia", "prioridade": "alta"}
    ),
    Document(
        page_content="""PROTOCOLO DE EXAME PREVENTIVO (PAPANICOLAU):
        Indicado para mulheres de 25 a 64 anos que já tiveram atividade sexual.
        Periodicidade: Após dois exames anuais negativos, realizar a cada 3 anos.
        Preparo: Não ter relações sexuais ou usar duchas/cremes vaginais 48h antes.""",
        metadata={"categoria": "ginecologia", "prioridade": "media"}
    ),
    Document(
        page_content="""PROTOCOLO DE INSERÇÃO DE DIU (COBRE E MIRENA):
        Indicação: Pacientes que desejam contracepção de longo prazo.
        Contraindicações: Gravidez confirmada, infecção pélvica ativa ou malformações uterinas.
        Pós-procedimento: É comum haver cólicas leves; acompanhamento com USG após 30 dias.""",
        metadata={"categoria": "planejamento_familiar", "prioridade": "media"}
    ),
    Document(
        page_content="""PROTOCOLO DE CLIMATÉRIO E MENOPAUSA:
        Definição: A menopausa é confirmada após 12 meses sem menstruação. O climatério é a transição.
        Sintomas: Ondas de calor (fogachos), suores noturnos, secura vaginal e mudanças de humor.
        Tratamento: Avaliação de Terapia de Reposição Hormonal (TRH) e mudanças no estilo de vida.
        Exames: Mamografia anual e Densitometria Óssea para monitorar a saúde dos ossos.""",
        metadata={"categoria": "ginecologia", "prioridade": "media"}
    )
]

print(f"✅ {len(protocolos_hospitalares)} protocolos prontos para vetorização!")

✅ 4 protocolos prontos para vetorização!


### Sessão 5: Vetorização e Criação do Banco de Dados (FAISS)
- Objetivo: Transformar textos em vetores e criar o índice de busca local.

In [11]:
# ==============================================================================
# SEÇÃO 5: VETORIZAÇÃO E ARMAZENAMENTO (FAISS)
# Objetivo: Transformar textos em vetores e criar o índice de busca local.
# ==============================================================================

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Inicializamos o modelo de Embeddings (o "tradutor" para números)
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# 2. Criamos o banco de dados FAISS a partir dos nossos documentos
vectorstore = FAISS.from_documents(protocolos_hospitalares, embeddings)

# 3. Configuramos o "Retriever" (o buscador)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2}) # Busca os 2 trechos mais relevantes

print("✅ Banco de dados vetorial criado e pronto para consultas!")

✅ Banco de dados vetorial criado e pronto para consultas!


In [12]:
# Passo 1: Identificar o modelo de Embedding correto

import google.generativeai as genai
import os
from dotenv import load_dotenv

load_dotenv()
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

print("Modelos de EMBEDDING disponíveis:")
try:
    for m in genai.list_models():
        if 'embedContent' in m.supported_generation_methods:
            print(f"ID: {m.name}")
except Exception as e:
    print(f"Erro: {e}")

Modelos de EMBEDDING disponíveis:
ID: models/gemini-embedding-001
ID: models/gemini-embedding-2-preview


In [13]:
# Passo 2: Ajustar a Célula de Vetorização

# ==============================================================================
# SEÇÃO 5: VETORIZAÇÃO E ARMAZENAMENTO (FAISS)
# Objetivo: Transformar textos em vetores e criar o índice de busca local.
# ==============================================================================

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Inicializamos o modelo de Embeddings correto (gemini-embedding-001)
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# 2. Criamos o banco de dados FAISS a partir dos nossos protocolos
vectorstore = FAISS.from_documents(protocolos_hospitalares, embeddings)

# 3. Configuramos o "Retriever" (o buscador)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print("✅ Banco de dados vetorial criado com gemini-embedding-001!")

✅ Banco de dados vetorial criado com gemini-embedding-001!


In [14]:
# Teste de Busca (Opcional, mas recomendado)
docs = retriever.invoke("Como devo me preparar para o preventivo?")
for doc in docs:
    print(f"Trecho encontrado: {doc.page_content[:100]}...")

Trecho encontrado: PROTOCOLO DE EXAME PREVENTIVO (PAPANICOLAU):
        Indicado para mulheres de 25 a 64 anos que já t...
Trecho encontrado: PROTOCOLO DE PRÉ-NATAL (BAIXO RISCO):
        As consultas devem ser mensais até a 28ª semana. 
    ...


### Sessão 6: Construção da Resposta Final (Cadeia RAG)
- Objetivo: Unir a busca com a geração de texto para responder à paciente.

In [15]:
# ==============================================================================
# SEÇÃO 6: GERAÇÃO DE RESPOSTA COM CONTEXTO (RAG)
# Objetivo: Unir a busca com a geração de texto para responder à paciente.
# ==============================================================================

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Definimos o Prompt que orienta como a IA deve usar o protocolo
template_resposta = """Você é um assistente de saúde gentil e profissional. 
Use APENAS os protocolos fornecidos abaixo para responder à pergunta da paciente.
Se não encontrar a resposta nos protocolos, diga que não possui essa informação específica e oriente a buscar um profissional.

PROTOCOLO RECUPERADO:
{context}

PERGUNTA DA PACIENTE:
{question}

RESPOSTA GENTIL:"""

prompt_rag = ChatPromptTemplate.from_template(template_resposta)

# 2. Criamos a cadeia RAG
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt_rag
    | llm
    | StrOutputParser()
)

print("✅ Cadeia RAG configurada e pronta!")

✅ Cadeia RAG configurada e pronta!


In [16]:
# Teste de Fogo

pergunta_clinica = "Quais exames preciso fazer no meu pré-natal?"
resposta = rag_chain.invoke(pergunta_clinica)

print(f"🤖 Assistente: {resposta}")

🤖 Assistente: Olá! É um prazer ajudar você com essa dúvida importante sobre o seu acompanhamento.

De acordo com o protocolo de pré-natal de baixo risco, os exames obrigatórios que você deve realizar são: Hemograma, Tipagem ABO/Rh, Glicemia, HIV e Sífilis.

Desejo que sua gestação seja muito tranquila. Caso tenha outras dúvidas, estou à disposição!


### Sessão 7: O Orquestrador (Unindo Roteador e RAG)
- Agora que temos as peças soltas, precisamos criar a lógica final. O objetivo é:
  - Receber a pergunta.
  - Passar pelo Roteador (Sessão 2).
  - Se for clinico, disparar a Cadeia RAG (Sessão 6).
  - Se for seguranca ou triagem, dar uma resposta de prioridade máxima.

In [17]:
# ==============================================================================
# SEÇÃO 7: ORQUESTRADOR INTERATIVO
# ==============================================================================

import time

def assistente_hospitalar(pergunta: str):
    # 1. O roteador decide o caminho
    classificacao = router_chain.invoke({"question": pergunta})
    topico = classificacao.topic
    
    print(f"[LOG] Tópico: {topico}") # Útil para você ver o robô "pensando"
    
    # 2. Execução da lógica baseada no tópico
    if topico == "clinico":
        return rag_chain.invoke(pergunta)
    elif topico == "triagem":
        return "🚨 ATENÇÃO: Seus sintomas indicam a necessidade de avaliação imediata. Por favor, vá ao pronto-socorro ou ligue 192."
    elif topico == "seguranca":
        return "💜 Você não está sozinha. Para ajuda segura, ligue 180 ou 190."
    else:
        return "🤖 Desculpe, sou um assistente especializado em saúde da mulher. Não posso ajudar com esse assunto específico."

# --- LOOP DE CONVERSA ---
print("✨ Assistente Virtual pronto! (Digite 'sair' para encerrar)")

while True:
    pergunta_usuario = input("Sua dúvida: ")
    
    if pergunta_usuario.lower() in ["sair", "parar", "exit"]:
        print("Encerrando atendimento. Cuide-se! 👋")
        break
        
    resposta = assistente_hospitalar(pergunta_usuario)
    print(f"Assistente: {resposta}\n" + "-"*30)

    time.sleep(2)

✨ Assistente Virtual pronto! (Digite 'sair' para encerrar)
[LOG] Tópico: clinico
Assistente: Olá! É um prazer ajudar você com essa dúvida.

De acordo com o protocolo de saúde, os principais sintomas associados ao climatério e à menopausa são:

*   Ondas de calor (conhecidas como fogachos);
*   Suores noturnos;
*   Secura vaginal;
*   Mudanças de humor.

Lembre-se que o climatério é o período de transição, enquanto a menopausa é confirmada após 12 meses sem menstruação. Caso precise de orientações sobre tratamento ou exames de rotina, recomendo que agende uma consulta com seu médico para uma avaliação personalizada.
------------------------------
[LOG] Tópico: seguranca
Assistente: 💜 Você não está sozinha. Para ajuda segura, ligue 180 ou 190.
------------------------------
[LOG] Tópico: fora_de_escopo
Assistente: 🤖 Desculpe, sou um assistente especializado em saúde da mulher. Não posso ajudar com esse assunto específico.
------------------------------
Encerrando atendimento. Cuide-se! 👋